# Credit Card Fraud Detection Using Support Vector Machine (SVM)

**Academic Year:** 2026 | **Batch:** 13  
**Department:** Computer Science and Engineering (AI & ML)  
**Roll Numbers:** 24P61A66G7, 24P61A66G8, 24P61A66G9  

---

### 1. Problem Statement
Identify potentially fraudulent credit card transactions based on transaction-related numerical features.

### 2. Dataset Description
- **Dataset:** Kaggle Credit Card Fraud Detection Dataset (MLG-ULB)
- **Total Rows:** 284,807 transactions
- **Columns:** 31 (Time, Amount, V1-V28 PCA features, and Class target)
- **Class Imbalance:** 99.828% Legitimate (0) vs. 0.172% Fraud (1)


In [ ]:
# Step 1: Import core machine learning libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    average_precision_score,
    precision_recall_curve,
)

print("Libraries imported successfully.")


In [ ]:
# Step 2: Load the Credit Card Fraud Dataset
df = pd.read_csv('creditcard.csv')
print('Dataset Shape:', df.shape)
print('Missing Values:', df.isnull().sum().sum())
print('\nClass Distribution:')
print(df['Class'].value_counts())
print(f'Fraud Ratio: {df["Class"].mean() * 100:.3f}%')
df.head()


### 3. Data Preprocessing & Train/Test Split
We separate features from labels and split the dataset into an 80% training set and a 20% test set using stratified sampling.


In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training Set: {len(X_train):,} rows ({y_train.sum()} frauds)')
print(f'Testing Set : {len(X_test):,} rows ({y_test.sum()} frauds)')

# Standardize features (Mean = 0, Std = 1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### 4. Support Vector Machine (SVM) Model Training
We configure the SVM with `class_weight='balanced'` so the optimization algorithm assigns higher misclassification penalties to the minority fraud class.


In [ ]:
# Train SVM Classifier with Balanced Class Weights
svm = LinearSVC(dual=False, class_weight='balanced', max_iter=2000, random_state=42)
svm.fit(X_train_scaled, y_train)
print('SVM Model successfully trained!')


### 5. Performance Evaluation
Evaluate on all 56,962 held-out test transactions.


In [ ]:
y_pred = svm.predict(X_test_scaled)
scores = svm.decision_function(X_test_scaled)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
ap = average_precision_score(y_test, scores)

print('=' * 60)
print('CONFUSION MATRIX:')
print(cm)
print(f'True Negatives  (TN): {tn:,}')
print(f'False Positives (FP): {fp:,}')
print(f'False Negatives (FN): {fn}')
print(f'True Positives  (TP): {tp} / {y_test.sum()} (Recall: {tp/y_test.sum()*100:.1f}%)')
print(f'Average Precision (AP): {ap:.4f}')
print('=' * 60)
print('\nCLASSIFICATION REPORT:')
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))


In [ ]:
# Plot Confusion Matrix and Precision-Recall Curve
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'], ax=ax[0])
ax[0].set_title(f'SVM Confusion Matrix (N = {len(y_test):,})')
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, scores)
ax[1].plot(recall, precision, color='#4f46e5', linewidth=2.5, label=f'SVM (AP = {ap:.4f})')
ax[1].set_title('Precision-Recall Curve')
ax[1].set_xlabel('Recall')
ax[1].set_ylabel('Precision')
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.show()


### 6. Testing New Unseen Transactions
Demonstrate how the model evaluates single new transactions.


In [ ]:
def test_transaction(name, time_s, amount_d, v_features):
    row = np.array([[time_s] + list(v_features) + [amount_d]])
    row_scaled = scaler.transform(row)
    pred = svm.predict(row_scaled)[0]
    score = svm.decision_function(row_scaled)[0]
    status = '🚨 FRAUD DETECTED' if pred == 1 else '✅ LEGITIMATE'
    print(f'Transaction: {name}')
    print(f'  Amount: ${amount_d:.2f} | Time: {time_s}s')
    print(f'  Margin Distance: {score:.4f} -> Result: {status}\n')

# Normal Transaction Test
test_transaction('Normal Supermarket Grocery', 42000, 38.50, [0.0]*28)

# Fraud Attack Test (Anomalous V14)
fraud_vec = [0.0]*13 + [-5.4] + [0.0]*14
fraud_vec[3] = 3.8
test_transaction('Card-Testing Micro Attack', 1400, 1.00, fraud_vec)
